In [4]:
# === MERGED CELL (Updated for Drive dataset) ===

# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# === Drive + Project paths setup ===

from pathlib import Path   # FIXED: import added

# Base folder in Drive where everything goes (datasets, checkpoints, metrics)
DRIVE_ROOT = Path("/content/drive/MyDrive/BrainMRI_resnet50")

# Automatically create base folder if missing
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Project root (same as DRIVE_ROOT)
PROJECT_ROOT = DRIVE_ROOT

# Dataset folder: if dataset is in Drive
DATA_ROOT = str(DRIVE_ROOT / "brainMri")

# Checkpoint & metrics directories
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
METRICS_DIR    = PROJECT_ROOT / "metrics"

# Create them if missing
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# Quick misc vars
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 4
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
NUM_CLASSES = len(CLASS_NAMES)

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("📁 DRIVE_ROOT:", DRIVE_ROOT)
print("📁 DATA_ROOT:", DATA_ROOT)
print("📁 CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("📁 METRICS_DIR:", METRICS_DIR)
print("🚀 DEVICE:", DEVICE)


In [6]:
# === DATASET SETUP (Runtime Version) ===

# 1) Upload kaggle.json manually (same as original)
from google.colab import files
uploaded = files.upload()  # upload kaggle.json here

# 2) Move kaggle.json to ~/.kaggle and set permissions
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# 3) Download dataset into RUNTIME (fastest option)
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p ./dataset_tmp

# 4) Unzip inside runtime
!unzip -o ./dataset_tmp/brain-tumor-mri-dataset.zip -d ./brainMri

# 5) Set DATA_ROOT to runtime folder
DATA_ROOT = "/content/brainMri"

# 6) CLASS NAMES (explicit, stable)
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
NUM_CLASSES = len(CLASS_NAMES)

print("📁 Dataset downloaded + extracted to:", DATA_ROOT)
print("📌 Classes:", CLASS_NAMES)
print("🎉 Ready to proceed.")


Streaming output truncated to the last 5000 lines.
  inflating: ./brainMri/Training/glioma/Tr-gl_0715.jpg  
  inflating: ./brainMri/Training/glioma/Tr-gl_0716.jpg  
  inflating: ./brainMri/Training/glioma/Tr-gl_0717.jpg  
  inflating: ./brainMri/Training/glioma/Tr-gl_0718.jpg  
  inflating: ./brainMri/Training/glioma/Tr-gl_0719.jpg  
  inflating: ./brainMri/Training/glioma/Tr-gl_0720.jpg  
  inflating: ./brainMri/Training/glioma/Tr-gl_0721.jpg  
  inflating: ./brainMri/Training/glioma/Tr-gl_0722.jpg  
  inflating: ./brainMri/Training/glioma/Tr-gl_0723.jpg  
  inflating: ./brainMri/Training/glioma/Tr-gl_0724.jpg  
  inflating: ./brainMri/Training/glioma/Tr-gl_0725.jpg  
  inflating: ./brainMri/Training/glioma/Tr-gl_0726.jpg  
  inflating: ./brainMri/Training/glioma/Tr-gl_0727.jpg  
  inflating: ./brainMri/Training/glioma/Tr-gl_0728.jpg  
  inflating: ./brainMri/Training/glioma/Tr-gl_0729.jpg  
  inflating: ./brainMri/Training/glioma/Tr-gl_0730.jpg  
  inflating: ./brainMri/Training/glio

In [7]:
# === OPTIMIZED INSTALL + SYSTEM CONFIG ===

# Install required libraries (same as original)
!pip -q install --upgrade pip setuptools wheel
!pip -q install torchmetrics==1.4.0.post0 timm==1.0.9 scikit-learn==1.5.2 \
                opencv-python-headless==4.10.0.84 einops==0.8.0 tqdm==4.66.5

import os, sys, torch

# --- PERFORMANCE IMPROVEMENTS ---
# Use more CPU threads (your original forced only 1 → slowed dataloading massively)
os.environ["OMP_NUM_THREADS"] = "4"
torch.set_num_threads(4)

# Enable cuDNN benchmark for faster GPU conv operations (safe because fixed-size images)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

# Do NOT force mp spawn mode — unnecessary for single-process training
# (this line caused extra overhead and slow startup; removed for performance)

print("Python:", sys.version)
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU-only")
print("CPU Threads:", torch.get_num_threads())
print("OMP Threads:", os.environ['OMP_NUM_THREADS'])

# timm is already installed above, but kept for safety (fast)
!pip -q install timm


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch: 2.9.0+cu126 | CUDA: 12.6
GPU: Tesla T4
CPU Threads: 4
OMP Threads: 4


In [8]:
# === OPTIMIZED DATASET + LOADER CELL ===
from glob import glob
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
import torch, numpy as np
from tqdm.auto import tqdm
from pathlib import Path

# --- TRANSFORMS (same augmentations, but faster pipeline) ---
base_train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])
base_test_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

# --- DATASET ---
class BrainMRIDataset(Dataset):
    def __init__(self, root, split="train"):
        self.samples = []
        self.train = (split == "train")

        actual = "Training" if split == "train" else "Testing"

        for idx, cls in enumerate(CLASS_NAMES):
            cls_dir = Path(root) / actual / cls
            files = sorted(glob(str(cls_dir/"**/*"), recursive=True))
            for p in files:
                if p.lower().endswith((".png",".jpg",".jpeg",".bmp",".tif",".tiff")):
                    self.samples.append((p, idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]

        # Fast: load with PIL (native in PyTorch transforms)
        img = Image.open(path).convert("RGB")

        tf = base_train_tf if self.train else base_test_tf
        img = tf(img)

        return img, label, path

# --- LOADERS ---
def make_loaders(root, bs=BATCH_SIZE):
    train_ds = BrainMRIDataset(root, "train")
    test_ds  = BrainMRIDataset(root, "test")

    # Compute class weights for sampler
    counts = np.zeros(len(CLASS_NAMES), dtype=np.int64)
    for _, y in train_ds.samples:
        counts[y] += 1

    class_weights = 1.0 / np.clip(counts, 1, None)
    sample_weights = [float(class_weights[y]) for _, y in train_ds.samples]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_ds), replacement=True)

    print("✅ Dataset ready | Train:", len(train_ds), "| Test:", len(test_ds))
    print("Train class counts:", {c:int(n) for c,n in zip(CLASS_NAMES, counts)})

    # DataLoader — optimized
    train_loader = DataLoader(
        train_ds,
        batch_size=bs,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=True,
        persistent_workers=True
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=bs,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True
    )

    return train_loader, test_loader, counts

train_loader, test_loader, class_counts = make_loaders(DATA_ROOT)


✅ Dataset ready | Train: 5712 | Test: 1311
Train class counts: {'glioma': 1321, 'meningioma': 1339, 'notumor': 1595, 'pituitary': 1457}


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [9]:
# === LOSS & TRAINING / EVAL HELPERS (optimized for T4 + preserves features) ===
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize
import json, numpy as np
from tqdm.auto import tqdm
import torch.nn.functional as F
import torch.nn as nn

# FocalCE (unchanged logic)
class FocalCE(nn.Module):
    def __init__(self, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    def forward(self, logits, targets):
        ce = self.ce(logits, targets)
        with torch.no_grad():
            pt = F.softmax(logits, dim=-1).gather(1, targets.view(-1,1)).clamp_(1e-6,1-1e-6).squeeze()
        focal = ((1-pt)**self.gamma).mean()
        return ce + 0.5 * focal

# TRAIN ONE EPOCH (supports AMP when you pass a GradScaler)
def train_one_epoch(model, loader, opt, loss_fn, epoch, scaler=None):
    """
    Trains model for one epoch.
    If `scaler` (torch.cuda.amp.GradScaler) is given, AMP is used.
    Returns average training loss.
    """
    model.train()
    total, n = 0.0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch} [train]", leave=True)

    for x, y, _ in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        opt.zero_grad(set_to_none=True)

        if scaler is not None:
            # AMP path
            with torch.cuda.amp.autocast():
                logits = model(x)
                loss = loss_fn(logits, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        else:
            # FP32 path
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            opt.step()

        batch_size = x.size(0)
        total += loss.item() * batch_size
        n += batch_size
        pbar.set_postfix(loss=f"{(total/max(n,1)):.4f}")

    return total / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, loss_fn, epoch=0):
    """
    Evaluate model on a loader.
    Returns: (avg_loss, acc, f1_macro, auc_macro, (y_true, y_pred, y_prob))
    Uses sklearn for final metrics (fast, reliable).
    """
    model.eval()
    total, n = 0.0, 0
    y_true, y_pred, y_prob = [], [], []
    pbar = tqdm(loader, desc=f"Epoch {epoch} [eval]", leave=True)

    for x, y, _ in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        logits = model(x)
        loss = loss_fn(logits, y)
        probs = F.softmax(logits, dim=-1)

        total += loss.item() * x.size(0)
        n += x.size(0)

        y_true.extend(y.cpu().tolist())
        y_pred.extend(probs.argmax(1).cpu().tolist())
        y_prob.append(probs.cpu().numpy())  # append batch

        pbar.set_postfix(loss=f"{(total/max(n,1)):.4f}")

    # concatenate batch probs
    if len(y_prob):
        y_prob = np.concatenate(y_prob, axis=0)
    else:
        y_prob = np.zeros((0, NUM_CLASSES))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Accuracy & F1
    acc = float(accuracy_score(y_true, y_pred)) if y_true.size else float('nan')
    f1 = float(f1_score(y_true, y_pred, average='macro')) if y_true.size else float('nan')

    # AUC (robust to small class issues)
    try:
        y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
        # If only one class present in y_true this will raise — handle that
        auc = float(roc_auc_score(y_true_bin, y_prob, average='macro', multi_class='ovr'))
    except Exception:
        auc = float('nan')

    return total / max(n,1), acc, f1, auc, (y_true, y_pred, y_prob)


In [10]:
# === MODEL BUILDER ===
import timm
import torch.nn as nn

def get_timm_model(model_name, num_classes=NUM_CLASSES):
    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=num_classes,
        in_chans=3   # final dataset uses 3-channel images
    )
    return model


In [11]:
# === OPTIMIZED TRAINING LOOP (AMP + no redundant eval) ===

from torch.cuda.amp import GradScaler, autocast

def train_baseline(model, model_name, epochs=20):
    model = model.to(DEVICE)
    PATIENCE = 8

    # Loss, optimizer, scheduler
    loss_fn = FocalCE(gamma=2.0, label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=3, factor=0.5
    )

    # Mixed precision scaler
    scaler = GradScaler()

    # History dict (unchanged)
    history = {
        "train_loss": [], "val_loss": [],
        "train_acc": [], "val_acc": [],
        "train_f1":  [], "val_f1":  [],
        "train_auc": [], "val_auc": []
    }

    best_auc = -1
    save_path = PROJECT_ROOT / "checkpoints" / f"{model_name}_best.pth"
    no_improve_epochs = 0

    for epoch in range(1, epochs + 1):
        print(f"\n===== {model_name} | Epoch {epoch}/{epochs} =====")
        model.train()
        total_loss, n = 0, 0

        # --- TRAIN LOOP (AMP ENABLED) ---
        for x, y, _ in tqdm(train_loader, desc=f"Epoch {epoch} [train]"):
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)

            with autocast():  # AMP for speed
                logits = model(x)
                loss = loss_fn(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item() * x.size(0)
            n += x.size(0)

        train_loss = total_loss / max(n, 1)

        # --- EVALUATE ON TRAIN/VAL (ONLY ONCE EACH) ---
        t_loss, t_acc, t_f1, t_auc, _ = evaluate(model, train_loader, loss_fn, epoch)
        val_loss, val_acc, val_f1, val_auc, _ = evaluate(model, test_loader, loss_fn, epoch)
        scheduler.step(val_loss)

        # Update history
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(t_acc)
        history["val_acc"].append(val_acc)
        history["train_f1"].append(t_f1)
        history["val_f1"].append(val_f1)
        history["train_auc"].append(t_auc)
        history["val_auc"].append(val_auc)

        # Save best model
        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), save_path)
            print(f"💾 Saved {model_name} best | AUC={val_auc:.4f} ACC={val_acc:.4f}")
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            print(f"⚠️ No improvement for {no_improve_epochs} epochs")

        # Early stopping
        if no_improve_epochs >= PATIENCE:
            print(f"⏹️ Early stopping at epoch {epoch}")
            break

    return history


In [12]:
# === TRAIN ResNet50 (run this cell in one runtime/account) ===
model_name = "ResNet50_runA"
model = get_timm_model("resnet50", num_classes=NUM_CLASSES)
history_resnet50 = train_baseline(model, model_name, epochs=20)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



===== ResNet50_runA | Epoch 1/20 =====


/tmp/ipython-input-1319036821.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch 1 [train]:   0%|          | 0/357 [00:00<?, ?it/s]

/tmp/ipython-input-1319036821.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # AMP for speed


...

Epoch 20 [train]:   0%|          | 0/357 [00:00<?, ?it/s]

/tmp/ipython-input-1319036821.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # AMP for speed


Epoch 20 [eval]:   0%|          | 0/357 [00:00<?, ?it/s]

Epoch 20 [eval]:   0%|          | 0/82 [00:00<?, ?it/s]

⚠️ No improvement for 7 epochs


In [13]:

# save history CSV to Drive
import pandas as pd
pd.DataFrame(history_resnet50).to_csv(METRICS_DIR / f"{model_name}_history.csv", index=False)
print("Saved history to:", METRICS_DIR / f"{model_name}_history.csv")
print("Best checkpoint (expected):", CHECKPOINT_DIR / f"{model_name}_best.pth")


Saved history to: /content/drive/MyDrive/BrainMRI/metrics/ResNet50_runA_history.csv
Best checkpoint (expected): /content/drive/MyDrive/BrainMRI/checkpoints/ResNet50_runA_best.pth
